# vLLM server on Colab

Serves a Qwen/Mistral/Llama checkpoint as an OpenAI-compatible endpoint and exposes it over ngrok,
so the agent running locally can call it. One profile selects the model, GPU budget and context
length together; nothing else needs editing.

Order of execution: GPU check → install → ngrok token → tunnel → anti-idle → server. The server cell
runs a heartbeat loop and is meant to stay running for the whole session.

Two install cells are provided and they are mutually exclusive:

| Install cell | vLLM | Use for |
|---|---|---|
| standard | `0.6.6` | Qwen2.5 (7b/14b/32b/72b), deepseek-32b, phi-4, llama-3.1 |
| recent | `>=0.8.0` | devstral-small, mistral-nemo, qwen3, qwen3.6 |

Once the tunnel is up, put the URL in `.env` as `THESIS_LLM_API_BASE=<url>/v1` and run tasks with
`bash scripts/run.sh <profile> <subset>`.


## 1. GPU and RAM

Pick the runtime before running anything else: T4 16GB serves 7b only, A100 40GB covers most
profiles, A100 80GB is required for 72b. Change it under Runtime → Change runtime type.


In [1]:
import psutil

gpu_info = !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null
if gpu_info and 'failed' not in '\n'.join(gpu_info).lower():
    print("GPU detected:")
    for line in gpu_info:
        print(f"   {line}")
else:
    print("No GPU. Runtime -> Change runtime type.")

# Report which profiles the available VRAM can actually serve.
vram_info = !nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits 2>/dev/null
if vram_info:
    vram_gb = int(vram_info[0].strip()) / 1024
    print(f"\nVRAM: {vram_gb:.1f} GB")
    if vram_gb >= 75:
        print("   A100 80GB  - serves: 7b, 14b, 32b, 32b-fp16, 72b")
    elif vram_gb >= 38:
        print("   A100 40GB  - serves: 7b, 14b, 32b, qwen3-32b, qwen3.6-27b")
        print("   72b requires A100 80GB")
    elif vram_gb >= 22:
        print("   L4 24GB    - serves: 7b, 14b (32b-AWQ marginal)")
        print("   32b, 72b require A100")
    elif vram_gb >= 14:
        print("   T4 16GB    - serves: 7b")
        print("   14b, 32b, 72b require A100")
    else:
        print("   Insufficient VRAM for any profile.")

ram_gb = psutil.virtual_memory().total / 1e9
print(f"\nRAM: {ram_gb:.1f} GB")
if ram_gb >= 12:
    print("   Sufficient.")
else:
    print("   Low. If loading stalls, switch to a High-RAM runtime.")


✅ GPU detected:
   NVIDIA L4, 23034 MiB, 22564 MiB

📊 VRAM: 22.5 GB
   ⚠️  L4 24GB   →  compatible: 7b, 14b (32b-AWQ marginal)
   ❌ 32b, 72b require A100

📊 RAM: 56.9 GB
   ✅ Αρκετή RAM


## 2. Install

Installs vLLM and pyngrok. `transformers` is pinned to 4.46.3 because 4.47 removed
`all_special_tokens_extended`, which vLLM 0.6.6 still calls. Takes 2-3 minutes; no runtime restart
needed. Model weights download on first server start, which adds a few more minutes.


In [ ]:
# Standard install: Qwen2.5, deepseek-32b, phi-4, llama-3.1.
# transformers is pinned because 4.47 dropped `all_special_tokens_extended`, which vLLM 0.6.6 calls.
print("Installing vllm==0.6.6, transformers==4.46.3, pyngrok ...")
!pip install -q "vllm==0.6.6" "transformers==4.46.3" pyngrok
print("Done.")


📦 Installing vLLM==0.6.6 + pinned transformers (T4-compatible) and pyngrok...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 143.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 153.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363

In [ ]:
# Recent install: devstral-small, mistral-nemo, qwen3, qwen3.6.
# These use architectures absent from transformers 4.46.3 (mistral3, qwen3), so they need a newer
# vLLM built against CUDA 12. Run this INSTEAD of the standard install cell, not after it.
# The wheel is ~300MB and Colab often times out on it, hence the extended timeout and retries.
print("Installing vllm>=0.8.0,<0.14.0, transformers>=4.52.0, pyngrok ...")
!pip install -q --timeout 300 --retries 5 "vllm>=0.8.0,<0.14.0" "transformers>=4.52.0" pyngrok
print("Done. Restart the runtime if prompted.")


📦 Installing vLLM 0.8–0.13 (CUDA 12-compatible) + transformers>=4.52 (Mistral-3.1 compatible)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.9/474.9 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 139.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Ngrok token

Get an authtoken from the ngrok dashboard (free tier is enough) and set it below. The HF token is
only needed for gated repositories.


In [ ]:
import os
from pyngrok import ngrok

NGROK_TOKEN = ""   # required — ngrok dashboard, "Your Authtoken"
HF_TOKEN    = ""   # optional — only for gated HuggingFace repositories

assert NGROK_TOKEN, "Set NGROK_TOKEN above."

ngrok.set_auth_token(NGROK_TOKEN)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("Ngrok token configured.")


✅ Ngrok token configured.


## 4. Tunnel

Opens the public HTTPS endpoint. Done before starting vLLM so the URL is available while the model
loads.


In [4]:
VLLM_PORT = 8000

# Opened before vLLM starts so the URL is known while the model loads.
tunnel = ngrok.connect(VLLM_PORT, "http")
public_url = tunnel.public_url

print("=" * 60)
print("NGROK PUBLIC URL")
print("=" * 60)
print(f"  {public_url}")
print("=" * 60)
print()
print("Add to .env on the local machine:\n")
print(f"  THESIS_LLM_API_BASE={public_url}/v1")
print()
print("Next: run the anti-idle cell, then the server cell.")


🌐  NGROK PUBLIC URL
  https://agrobiologic-firefly-ezekiel.ngrok-free.dev

📋  Στο .env στον υπολογιστή σου, βάλε:

  THESIS_LLM_API_BASE=https://agrobiologic-firefly-ezekiel.ngrok-free.dev/v1

📋  Ή export στο terminal:

  export THESIS_LLM_PROVIDER=vllm
  export THESIS_LLM_API_BASE="https://agrobiologic-firefly-ezekiel.ngrok-free.dev/v1"

Μετά τρέξε:  THESIS_MODEL_PROFILE=32b bash scripts/run_single.sh hello-world

⏳  Τώρα τρέξε το anti-idle κελί (11) και μετά το vLLM κελί (12).


## 5. Server

Set `MODEL_PROFILE` in the cell below. Each profile fixes the checkpoint, quantization, context
length and GPU utilisation, and declares which install cell it needs.

**Standard install cell (vLLM 0.6.6):** `7b`, `14b`, `32b`, `72b`, `deepseek-32b`, `phi-4`,
`llama-3.1-8b`, `llama-3.1-70b`

**Recent install cell (vLLM >= 0.8.0):** `devstral-small`, `mistral-nemo-12b`, `qwen3-32b`,
`qwen3.6-27b`

`72b` needs an A100 80GB; everything else fits on A100 40GB, and `7b` also fits a T4. The cell
verifies available VRAM against the profile before launching and stops early if it is insufficient.

Run the anti-idle cell first — Colab disconnects idle runtimes, and a long sweep will otherwise be
cut short.


In [5]:
# Keeps the Colab session alive for the duration of a sweep. Two independent mechanisms, because
# either alone is unreliable: an AudioContext stops the browser throttling a backgrounded tab, and
# a periodic synthetic event keeps Colab's own idle timer from firing. Server-side idle detection is
# handled separately by the heartbeat print loop in the server cell.
# Run once, before the server cell.
from IPython.display import display, Javascript

display(Javascript("""
(function() {
  // Browsers do not suspend a tab holding an active AudioContext.
  try {
    if (window._keepAliveAudio) {
      window._keepAliveAudio.close();
    }
    const ctx = new AudioContext();
    const oscillator = ctx.createOscillator();
    const gain = ctx.createGain();
    gain.gain.value = 0.00001;  // inaudible, but must be non-zero
    oscillator.frequency.value = 440;
    oscillator.connect(gain);
    gain.connect(ctx.destination);
    oscillator.start();
    window._keepAliveAudio = ctx;
    console.log('keepalive: AudioContext running');
  } catch(e) {
    console.warn('keepalive: AudioContext unavailable, using interval only', e);
  }

  // Fallback: synthetic activity every 30s.
  function simulateActivity() {
    const connect = document.querySelector('colab-connect-button');
    if (connect) connect.click();
    document.dispatchEvent(new MouseEvent('mousemove', {bubbles: true}));
    document.dispatchEvent(new KeyboardEvent('keydown', {bubbles: true, key: 'Shift'}));
  }

  if (window._keepAliveInterval) clearInterval(window._keepAliveInterval);
  window._keepAliveInterval = setInterval(simulateActivity, 30000);

  console.log('keepalive: activity simulation every 30s');
})();
"""))

print("Anti-idle active. The tab can be backgrounded without the runtime disconnecting.")


<IPython.core.display.Javascript object>

✅ Anti-idle ενεργό (Web Audio API + activity simulation).
   Το tab δεν θα κλείσει ακόμα και αν πάει σε background.
   Μπορείς να αλλάξεις tab ελεύθερα.


In [ ]:
import subprocess, time, sys, os, requests

MODEL_PROFILE = "mistral-nemo-12b"

# No seed is set: at temperature 0 the sampling RNG is never consulted, so --seed is a no-op in vLLM.
# The residual run-to-run variance comes from CUDA non-determinism in quantized kernels and is
# reported through bootstrap intervals rather than suppressed here.
ENABLE_NATIVE_TOOL_CALLING = False

# Each profile fixes the checkpoint, quantization, context length and GPU utilisation, plus the
# minimum VRAM it needs and, where relevant, the vLLM version it requires.
# For reference at fp16: 7B ~14GB, 14B ~28GB, 24B ~48GB, 32B ~64GB, 70B ~140GB. AWQ 4-bit brings
# these to roughly a quarter.
PROFILES = {
    "7b": {
        "model":       "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ",
        "served_as":   "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ",
        "quantization": "awq",
        "max_model_len": 16384,
        "gpu_util":    0.92,
        "tool_parser": "hermes",
        "min_vram_gb": 14,    # 7B AWQ ≈ 4GB weights + KV cache
    },
    "14b": {
        "model":       "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ",
        "served_as":   "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ",
        "quantization": "awq",
        "max_model_len": 32768,
        "gpu_util":    0.92,
        "tool_parser": "hermes",
        "min_vram_gb": 22,    # 14B AWQ ≈ 8GB weights + KV cache
    },
    "32b": {
        "model":       "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ",
        "served_as":   "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ",
        "quantization": "awq",
        "max_model_len": 32768,
        "gpu_util":    0.92,
        "tool_parser": "hermes",
        "min_vram_gb": 38,    # 32B AWQ ≈ 18GB weights + KV cache
    },
    "72b": {
        "model":       "Qwen/Qwen2.5-72B-Instruct-AWQ",
        "served_as":   "Qwen/Qwen2.5-72B-Instruct-AWQ",
        "quantization": "awq",
        "max_model_len": 32768,
        "gpu_util":    0.90,
        "tool_parser": "hermes",
        "min_vram_gb": 75,    # 72B AWQ ≈ 40GB — needs 80GB GPU
    },
    "mistral-nemo-12b": {
        "model":       "casperhansen/mistral-nemo-instruct-2407-awq",
        "served_as":   "casperhansen/mistral-nemo-instruct-2407-awq",
        "quantization": "awq",      # AWQ 4-bit (bartowski) — standard, no VLM issues
        "max_model_len": 32768,
        "gpu_util":    0.90,
        "tool_parser": "mistral",
        "tokenizer_mode": "mistral",  # Mistral-Nemo uses tekken.json tokenizer
        "tokenizer":   "mistralai/Mistral-Nemo-Instruct-2407",
        "min_vram_gb": 14,    # 12B AWQ ≈ 6GB weights + KV cache — fits any A100
        "requires_vllm": ">=0.8.0",  # mistral arch — use Mistral install cell
    },
    "deepseek-32b": {
        "model":       "casperhansen/deepseek-r1-distill-qwen-32b-awq",
        "served_as":   "casperhansen/deepseek-r1-distill-qwen-32b-awq",
        "quantization": "awq",      # AWQ 4-bit (188k downloads, MIT, casperhansen)
        "max_model_len": 32768,
        "gpu_util":    0.90,
        "tool_parser": "hermes",
        "min_vram_gb": 38,    # 32B AWQ ≈ 16GB weights + KV cache — A100 40GB ✓
    },
    "devstral-small": {
        "model":       "cyankiwi/Devstral-Small-2-24B-Instruct-2512-AWQ-4bit",
        "served_as":   "cyankiwi/Devstral-Small-2-24B-Instruct-2512-AWQ-4bit",
        "quantization": "compressed-tensors",  # cyankiwi packs AWQ-int4 as compressed-tensors (pack-quantized W4 g32) — 24B ≈ 12-14GB → A100 40GB ✓
        "max_model_len": 32768,     # native 256K — capped at 32K for KV-fit + grid parity
        "gpu_util":    0.90,
        "tool_parser": "mistral",
        "tokenizer_mode": "mistral",  # mistral3 arch (Devstral-Small-2 = Mistral-Small-3.x base)
        "tokenizer":   "mistralai/Devstral-Small-2-24B-Instruct-2512",  # tekken tokenizer
        "min_vram_gb": 22,    # 24B AWQ ≈ 12-14GB weights + KV cache — A100 40GB ✓
        "requires_vllm": ">=0.8.0",  # mistral3 + Llama4 rope — needs RECENT vLLM (Dec-2025; card recommends nightly)
    },
    "phi-4": {
        "model":       "curiousmind147/microsoft-phi-4-AWQ-4bit-GEMM",
        "served_as":   "curiousmind147/microsoft-phi-4-AWQ-4bit-GEMM",
        "quantization": "awq",      # AWQ 4-bit — 14B ≈ 7GB weights → A100 40GB ✓
        "max_model_len": 16384,     # phi-4 max context is 16K (not 32K like Qwen)
        "gpu_util":    0.90,
        "tool_parser": "hermes",    # phi-4 uses chatml format
        "min_vram_gb": 14,    # 14B AWQ ≈ 7GB weights + KV cache — any A100 ✓
        "requires_vllm": "==0.6.6",  # phi3 arch — use standard install cell
    },
    "llama-3.1-8b": {
        "model":       "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
        "served_as":   "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
        "quantization": "awq",      # AWQ 4-bit — 8B ≈ 4GB weights → L4 22GB ✓
        "max_model_len": 32768,
        "gpu_util":    0.90,
        "tool_parser": "llama3_json",
        "min_vram_gb": 8,     # 8B AWQ ≈ 4GB weights → fits L4 22GB comfortably
        "requires_vllm": "==0.6.6",  # llama arch — use standard install cell
    },
    "llama-3.1-70b": {
        "model":       "hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4",
        "served_as":   "hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4",
        "quantization": "awq",      # AWQ 4-bit — 70B ≈ 35GB weights → A100 40GB (tight)
        "max_model_len": 8192,      # Reduced to fit KV cache on 40GB alongside 35GB weights
        "gpu_util":    0.97,        # Push limit — 35GB weights leaves ~5GB for KV cache
        "tool_parser": "llama3_json",
        "min_vram_gb": 38,    # 70B AWQ ≈ 35GB weights + KV cache — A100 40GB (tight) ✓
        "requires_vllm": "==0.6.6",  # llama arch — use standard install cell
    },
    "qwen3-32b": {
        "model":       "Qwen/Qwen3-32B-AWQ",
        "served_as":   "Qwen/Qwen3-32B-AWQ",
        "quantization": "awq",      # AWQ 4-bit — 32B ≈ 18-20GB weights → A100 40GB ✓
        "max_model_len": 32768,
        "gpu_util":    0.90,
        "tool_parser": "hermes",    # Qwen3 uses Hermes-style tool format
        "min_vram_gb": 38,    # 32B AWQ ≈ 18-20GB weights + KV cache — A100 40GB ✓
        "requires_vllm": ">=0.8.0",  # Qwen3 arch ≥0.8.5; non-thinking flag needs ≥0.9.0 → Mistral install cell
        # Qwen3 is a HYBRID reasoning model (thinking ON by default). For a fair
        # comparison vs non-reasoning Qwen2.5 we serve it NON-THINKING server-wide:
        "reasoning_parser": "qwen3",
    },
    "qwen3.6-27b": {
        "model":       "cyankiwi/Qwen3.6-27B-AWQ-INT4",
        "served_as":   "cyankiwi/Qwen3.6-27B-AWQ-INT4",
        "quantization": "awq",      # AWQ-INT4 — 27B ≈ 16-21GB weights → A100 40GB ✓
        "max_model_len": 32768,     # native ctx 262K — capped at 32K for KV-fit + grid parity
        "gpu_util":    0.90,
        "tool_parser": "qwen3_coder",  # Qwen3.6 agentic-coding tool format (only used if native tools ON)
        "min_vram_gb": 38,    # 27B AWQ ≈ 16-21GB weights + KV cache — A100 40GB ✓
        "requires_vllm": ">=0.8.0",  # Qwen3.6 arch (Apr-2026) needs a RECENT vLLM (newer than Qwen3-32B floor) → Mistral install cell
        # Latest dense open agentic-coding flagship that fits 40GB (TB2.0=59.3, SWE-bench 77.2).
        # HYBRID reasoning model — served NON-THINKING server-wide for fairness:
        "reasoning_parser": "qwen3",
        "enable_thinking": False,
    },
    "qwen3-30b": {
        "model":       "Qwen/Qwen3-Coder-30B-A3B-Instruct",
        "served_as":   "qwen3-coder-30b",
        "quantization": None,       # fp16 MoE — no AWQ for MoE arch → needs 80GB
        "max_model_len": 32768,
        "gpu_util":    0.88,
        "tool_parser": "hermes",
        "min_vram_gb": 65,    # 60GB weights + KV cache — requires A100 80GB
    },
    "llama-70b": {
        "model":       "meta-llama/Llama-3.3-70B-Instruct",
        "served_as":   "llama-3.3-70b",
        "quantization": None,       # fp16 — 70B×2bytes = 140GB → needs 2×A100 80GB (GENCI only)
        "max_model_len": 32768,
        "gpu_util":    0.90,
        "tool_parser": "llama3_json",
        "min_vram_gb": 150,   # NOT available on Colab — use GENCI 2×A100-80GB
    },
}

assert MODEL_PROFILE in PROFILES, \
    f"Unknown profile '{MODEL_PROFILE}'. Choose from: {list(PROFILES.keys())}"
prof = PROFILES[MODEL_PROFILE]

# The two install cells are not interchangeable; warn if the profile needs the other one.
requires_vllm = prof.get("requires_vllm", None)
if requires_vllm == ">=0.8.0":
    print("This profile requires vLLM >= 0.8.0. Use the recent install cell, not vllm==0.6.6.")
elif requires_vllm == "==0.6.6":
    print("This profile uses the standard vllm==0.6.6 install cell.")

# Fail before launching rather than after a long download.
vram_info = !nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits 2>/dev/null
if vram_info:
    available_vram = int(vram_info[0].strip()) / 1024
    if available_vram < prof["min_vram_gb"]:
        print(f"Not enough VRAM for profile '{MODEL_PROFILE}'!")
        print(f"   Available: {available_vram:.0f}GB  |  Needed: {prof['min_vram_gb']}GB")
        if MODEL_PROFILE in ("mistral-nemo-12b", "deepseek-32b", "devstral-small", "phi-4", "llama-3.1-8b", "llama-3.1-70b", "qwen3-30b", "qwen3-32b", "qwen3.6-27b"):
            print(f"   Switch to an A100 40GB runtime:")
            print(f"     Runtime → Change runtime type → A100 (requires Colab Pro+)")
        elif MODEL_PROFILE == "llama-70b":
            print(f"   llama-70b does not fit on Colab; it needs 2x A100-80GB.")
        else:
            print(f"   Switch to a smaller profile or a larger runtime.")
        raise SystemExit(1)
    print(f"VRAM check passed: {available_vram:.0f}GB available "
          f"(need {prof['min_vram_gb']}GB)")

MODEL_NAME    = prof["model"]
SERVED_AS     = prof["served_as"]
MAX_MODEL_LEN = prof["max_model_len"]
QUANTIZATION  = prof["quantization"]
GPU_UTIL      = prof["gpu_util"]
TOOL_PARSER   = prof["tool_parser"]

# Build vLLM command
cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--dtype", "float16",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_UTIL),
    "--port", str(VLLM_PORT),
    "--served-model-name", SERVED_AS,
]
if QUANTIZATION:
    cmd.extend(["--quantization", QUANTIZATION])
TOKENIZER_MODE = prof.get("tokenizer_mode", None)
TOKENIZER_NAME = prof.get("tokenizer", None)
if TOKENIZER_MODE:
    cmd.extend(["--tokenizer-mode", TOKENIZER_MODE])
if TOKENIZER_NAME:
    cmd.extend(["--tokenizer", TOKENIZER_NAME])

# Qwen3 emits <think> blocks by default. Non-thinking is forced server-wide so these models are
# compared on the same terms as the rest of the grid. Requires vLLM >= 0.9.0.
REASONING_PARSER = prof.get("reasoning_parser", None)
ENABLE_THINKING  = prof.get("enable_thinking", None)  # None = leave model default
if REASONING_PARSER:
    cmd.extend(["--reasoning-parser", REASONING_PARSER])
if ENABLE_THINKING is False:
    cmd.extend(["--default-chat-template-kwargs", '{"enable_thinking": false}'])

if ENABLE_NATIVE_TOOL_CALLING:
    cmd.extend(["--enable-auto-tool-choice", "--tool-call-parser", TOOL_PARSER])

vllm_proc = subprocess.Popen(
    cmd,
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)

print(f"vLLM started (PID {vllm_proc.pid})")
print(f"   Profile:       {MODEL_PROFILE}")
print(f"   Model:         {MODEL_NAME}")
print(f"   Served as:     {SERVED_AS}")
print(f"   Quantization:  {QUANTIZATION or 'fp16'}")
print(f"   max_model_len: {MAX_MODEL_LEN}")
print(f"   Tool parser:   {TOOL_PARSER}")
print(f"   Thinking:      {'OFF (non-thinking)' if prof.get('enable_thinking') is False else 'model default'}")
print(f"   Temperature:   0.0 (greedy — no seed needed)")
print(f"   Native tools:  {'enabled' if ENABLE_NATIVE_TOOL_CALLING else 'disabled (prompt-based)'}")
print(f"Logs: /tmp/vllm.log\n")
print(f"Loading model; weights download on first run.\n")

# Poll until the API answers, up to 15 minutes for a cold weight download.
for i in range(180):  # up to 15 min
    time.sleep(5)
    try:
        r = requests.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=2)
        if r.status_code == 200:
            print(f"Server up after {(i+1)*5}s!")
            print(f"   Models: {[m['id'] for m in r.json()['data']]}")
            os.system("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader 2>/dev/null")
            break
    except Exception:
        pass
    if (i + 1) % 6 == 0:
        elapsed = (i + 1) * 5
        print(f"Still loading... ({elapsed}s elapsed)")
        if vllm_proc.poll() is not None:
            print("vLLM process exited. Log follows:")
            os.system("cat /tmp/vllm.log")
            break
else:
    print("Timed out waiting for the server. Log follows:")
    os.system("cat /tmp/vllm.log")

print()
print("=" * 60)
print(f"Add to .env on the local machine:")
print(f"    THESIS_LLM_API_BASE={public_url}/v1")
print(f"    THESIS_MODEL_PROFILE={MODEL_PROFILE}")
print()
print(f"  Then run:")
print(f"    THESIS_MODEL_PROFILE={MODEL_PROFILE} bash scripts/run.sh")
print("=" * 60)

# Periodic output keeps the server-side idle timer from disconnecting the runtime, and doubles as
# a liveness check on the vLLM process.
print("\nHeartbeat active; one line per minute.\n")
heartbeat = 0
start_ts = time.time()
ntc_tag = f" | native tools: {TOOL_PARSER}" if ENABLE_NATIVE_TOOL_CALLING else ""
while True:
    time.sleep(60)
    heartbeat += 1
    uptime_min = (time.time() - start_ts) / 60

    if vllm_proc.poll() is not None:
        print(f"[{uptime_min:.0f}min] vLLM process exited. See /tmp/vllm.log")
        os.system("tail -20 /tmp/vllm.log")
        break

    try:
        r = requests.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=3)
        api_status = "API OK" if r.status_code == 200 else f"HTTP {r.status_code}"
    except Exception as e:
        api_status = f"unreachable: {e}"

    try:
        vram = os.popen("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader,nounits 2>/dev/null").read().strip()
        vram_str = f"VRAM: {vram} MiB" if vram else "VRAM: N/A"
    except Exception:
        vram_str = ""

    print(f"[{uptime_min:.0f}min] {api_status} | {vram_str} | profile: {MODEL_PROFILE}{ntc_tag} | ngrok: {public_url}")



✅ VRAM check passed: 22GB available (need 14GB)
🚀 vLLM started (PID 3868)
   Profile:       mistral-nemo-12b
   Model:         casperhansen/mistral-nemo-instruct-2407-awq
   Served as:     casperhansen/mistral-nemo-instruct-2407-awq
   Quantization:  awq
   max_model_len: 32768
   Tool parser:   mistral
   Temperature:   0.0 (greedy — no seed needed)
   Native tools:  ❌ disabled (prompt-based)
📄 Logs: /tmp/vllm.log

⏳ Loading model (first time downloads weights)...

⏳ Still loading... (30s elapsed)
⏳ Still loading... (60s elapsed)
⏳ Still loading... (90s elapsed)
⏳ Still loading... (120s elapsed)
⏳ Still loading... (150s elapsed)
⏳ Still loading... (180s elapsed)
⏳ Still loading... (210s elapsed)
⏳ Still loading... (240s elapsed)
⏳ Still loading... (270s elapsed)
⏳ Still loading... (300s elapsed)
✅ vLLM server is UP after 310s!
   Models: ['casperhansen/mistral-nemo-instruct-2407-awq']

📋  Στο PC σου, βάλε στο .env:
    THESIS_LLM_API_BASE=https://agrobiologic-firefly-ezekiel.ngrok-fre

### Debugging a failed start

Run these only if the server did not come up: CUDA version and libraries, then errors and the head
of the vLLM log.


In [8]:
# CUDA toolkit version and runtime libraries.
!nvcc --version
!ls /usr/local/cuda*/lib64/libcudart.so*


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
/usr/local/cuda-12.8/lib64/libcudart.so
/usr/local/cuda-12.8/lib64/libcudart.so.12
/usr/local/cuda-12.8/lib64/libcudart.so.12.8.90
/usr/local/cuda-12/lib64/libcudart.so
/usr/local/cuda-12/lib64/libcudart.so.12
/usr/local/cuda-12/lib64/libcudart.so.12.8.90
/usr/local/cuda/lib64/libcudart.so
/usr/local/cuda/lib64/libcudart.so.12
/usr/local/cuda/lib64/libcudart.so.12.8.90


In [9]:
# Errors reported by vLLM during startup.
!grep -n "Error\|Exception\|Cannot\|Unsupported\|Failed\|not support" /tmp/vllm.log | head -40


16:(APIServer pid=1683)     raise ValueError("No tokenizer file found.")
17:(APIServer pid=1683) ValueError: No tokenizer file found.
75:(APIServer pid=1683)     raise ValueError(f"No valid tokenizer file found in the repo {repo_id}.")
76:(APIServer pid=1683) ValueError: No valid tokenizer file found in the repo casperhansen/mistral-nemo-instruct-2407-awq.


In [10]:
# Start of the log, where architecture and quantization mismatches surface.
!head -120 /tmp/vllm.log


2026-06-06 09:29:40.354721: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-06 09:29:40.425069: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(APIServer pid=1683) INFO 06-06 09:29:50 [api_server.py:1351] vLLM API server version 0.13.0
(APIServer pid=1683) INFO 06-06 09:29:50 [utils.py:253] non-default args: {'model': 'casperhansen/mistral-nemo-instruct-2407-awq', 'tokenizer_mode': 'mistral', 'dtype': 'float16', 'max_model_len': 32768, 'quantization': 'awq', 'served_model_name': ['casperhansen/mistral-nemo-instr

## 6. Connection test (optional)

Confirms the endpoint answers. Run it from a second tab, or locally by replacing `localhost:8000`
with the ngrok URL.


In [ ]:
from openai import OpenAI

TEST_URL = f"http://localhost:{VLLM_PORT}/v1"
client = OpenAI(base_url=TEST_URL, api_key="EMPTY")

# SERVED_AS is set by the server cell; the request must use the same name vLLM serves under.
resp = client.chat.completions.create(
    model=SERVED_AS,
    messages=[{"role": "user", "content": "Say 'API OK' and nothing else."}],
    max_tokens=10,
    temperature=0,
)
print(f"{MODEL_PROFILE}: {resp.choices[0].message.content}")


---
## Local side

Put the tunnel URL in `.env` at the project root:

```
THESIS_LLM_API_BASE=https://<your-ngrok-url>.ngrok-free.app/v1
THESIS_LLM_PROVIDER=vllm
```

Then run against it, passing the profile explicitly:

```bash
bash scripts/run.sh 32b core_64                    # full task set
bash scripts/run.sh 32b single:hello-world         # one task
```

The profile name given to `run.sh` must match `MODEL_PROFILE` above, since the two select the same
checkpoint from opposite ends of the connection.
